In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Groundedness Checking for RAG Answers

## 1. Project Overview

**Task:** Build a system that verifies whether LLM-generated answers are **truly supported** by the retrieved passages — detecting hallucinations, unsupported claims, and fabricated citations.

**Why this matters:** LLMs confidently generate plausible-sounding text even when the retrieved context doesn't support the claim. In RAG systems, this creates a dangerous gap: the user trusts the answer because it cites sources, but the cited passages may not actually say what the answer claims. Groundedness checking catches this.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for answer generation and groundedness verification
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **groundedness** — what it means for an answer to be supported by evidence |
| 2 | Build a **claim extractor** that breaks answers into atomic, verifiable claims |
| 3 | Implement a **claim-level verifier** that checks each claim against source passages |
| 4 | Design an **entailment scorer** — supported / partially supported / not supported |
| 5 | Detect **hallucination types**: fabricated facts, wrong citations, extrapolation |
| 6 | Build a **full groundedness pipeline** with scoring and flagging |

## 3. The Groundedness Problem

RAG systems can hallucinate in several ways, even with good retrieval:

| Hallucination Type | Example | How It Happens |
|--------------------|---------|----------------|
| **Fabricated fact** | "Redis supports up to 1 billion keys" | LLM invents a specific number not in any passage |
| **Wrong citation** | "According to [P03], Docker uses..." | Passage P03 is about databases, not Docker |
| **Extrapolation** | "This makes Redis faster than Memcached" | Passage mentions Redis speed but never compares to Memcached |
| **Conflation** | "Kubernetes Pods use Docker Compose" | Mixes concepts from two different passages incorrectly |
| **Temporal fabrication** | "Since 2024, K8s supports..." | Date not mentioned anywhere in context |

### Groundedness vs Correctness

```
GROUNDED + CORRECT:   Answer says X (source says X, X is true)       ✓ ideal
GROUNDED + INCORRECT: Answer says X (source says X, but X is wrong)  ~ source problem
UNGROUNDED + CORRECT: Answer says Y (source doesn't say Y, Y is true) ⚠ hallucination
UNGROUNDED + INCORRECT: Answer says Z (source doesn't say Z, Z is wrong) ✗ worst case
```

**Groundedness checking** only verifies whether the answer is supported by the sources — not whether the sources themselves are correct. This is the achievable, automatable check.

## 4. Pipeline Architecture

```
  Question → Retriever → Passages → LLM → Raw Answer
                                              │
                                    ┌─────────┴──────────┐
                                    │  GROUNDEDNESS CHECK │
                                    │                     │
                                    │  1. Extract claims  │
                                    │     from answer     │
                                    │                     │
                                    │  2. For each claim: │
                                    │     Check if any    │
                                    │     passage entails │
                                    │     this claim      │
                                    │                     │
                                    │  3. Score:          │
                                    │     SUPPORTED       │
                                    │     PARTIAL         │
                                    │     NOT SUPPORTED   │
                                    │                     │
                                    │  4. Overall verdict │
                                    │     + flagged claims│
                                    └─────────┬──────────┘
                                              │
                                     Verified Answer
                                     (with confidence)
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports

## 7. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter, defaultdict

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_ANSWER = 0.3     # Slightly high to encourage some hallucinations for testing
TEMP_VERIFY = 0.0     # Deterministic for verification

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Answer temp: {TEMP_ANSWER} (higher to test hallucination detection)")
print(f"Verify temp: {TEMP_VERIFY} (deterministic for judgment)")

## 8. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Knowledge Base & RAG Pipeline

## 9. Document Corpus

25 technical passages. We'll use the LLM to generate answers from these, then verify whether the answers are grounded.

In [ ]:
PASSAGES = [
    {"id": "P01", "topic": "docker",
     "text": "Docker containers package applications with their dependencies into isolated units that share the host OS kernel. Docker images are built from Dockerfiles using a layered file system. Each Dockerfile instruction creates a new immutable layer. Containers start in seconds and consume megabytes of memory, compared to virtual machines which require gigabytes and boot in minutes."},
    {"id": "P02", "topic": "kubernetes",
     "text": "Kubernetes orchestrates containerized applications across clusters. A Pod is the smallest deployable unit containing one or more containers sharing network and storage. Deployments manage desired state — specify 3 replicas and K8s ensures exactly 3 pods run. Services provide stable network endpoints. Horizontal Pod Autoscaler adjusts replicas based on CPU utilization."},
    {"id": "P03", "topic": "databases",
     "text": "PostgreSQL uses MVCC (Multi-Version Concurrency Control) so readers never block writers. Write-Ahead Logging ensures durability by recording changes before applying them. The default index type is B-tree, suitable for equality and range queries. PostgreSQL supports JSON, arrays, and full-text search natively."},
    {"id": "P04", "topic": "databases",
     "text": "Database sharding partitions data across multiple instances. Hash-based sharding distributes rows by hashing the shard key. Range-based sharding partitions by value ranges. Cross-shard queries require scatter-gather operations. Adding new shards requires careful data rebalancing to maintain even distribution."},
    {"id": "P05", "topic": "caching",
     "text": "Redis is an in-memory data structure store achieving sub-millisecond latency for read and write operations. It supports strings, lists, sets, sorted sets, and hashes. Redis provides RDB point-in-time snapshots and AOF append-only file persistence. Redis Sentinel handles automatic failover for high availability."},
    {"id": "P06", "topic": "caching",
     "text": "Cache invalidation strategies: TTL (time-to-live) automatically expires entries. Write-through updates cache and database simultaneously. Write-behind batches database writes asynchronously. Cache-aside loads data into cache only on a miss. The cache stampede problem occurs when many requests simultaneously encounter an expired popular key."},
    {"id": "P07", "topic": "api",
     "text": "REST APIs use HTTP methods: GET reads resources, POST creates, PUT replaces, PATCH partially updates, DELETE removes. URIs identify resources as nouns. Responses use HTTP status codes: 200 OK, 201 Created, 400 Bad Request, 404 Not Found, 500 Internal Server Error. JSON is the standard response format."},
    {"id": "P08", "topic": "api",
     "text": "GraphQL exposes a single endpoint where clients specify exact data requirements through typed queries. Schemas define types, fields, and relationships. Resolvers fetch data for each field. This eliminates over-fetching (getting too much data) and under-fetching (needing multiple requests). The N+1 query problem is solved using DataLoader batching."},
    {"id": "P09", "topic": "security",
     "text": "OAuth 2.0 enables third-party authorization without exposing user credentials. The authorization code flow: user is redirected to auth server, grants permission, receives a code, which is exchanged for an access token. Refresh tokens allow obtaining new access tokens. Access tokens typically expire after 1 hour."},
    {"id": "P10", "topic": "security",
     "text": "TLS encrypts data in transit using a handshake that establishes a shared secret via asymmetric cryptography, then switches to faster symmetric encryption for data transfer. Server certificates verify identity. Certificate authorities (CAs) sign certificates. Let's Encrypt provides free automated certificates."},
    {"id": "P11", "topic": "security",
     "text": "SQL injection occurs when user input is concatenated into SQL queries without sanitization. Parameterized queries prevent injection by separating SQL logic from data. Cross-site scripting (XSS) injects malicious scripts into web pages. Content Security Policy (CSP) headers restrict which scripts can execute."},
    {"id": "P12", "topic": "messaging",
     "text": "Apache Kafka is a distributed event streaming platform. Producers publish messages to topics partitioned across brokers for parallel processing. Consumer groups enable parallel consumption — each partition is assigned to exactly one consumer in the group. Messages are retained for a configurable period (default 7 days) regardless of consumption."},
    {"id": "P13", "topic": "messaging",
     "text": "RabbitMQ implements the AMQP protocol for message queuing. Exchanges route messages to queues based on routing keys and binding rules. Direct exchanges route by exact key match. Topic exchanges support wildcard patterns. Dead letter exchanges capture messages that fail processing after maximum retry attempts."},
    {"id": "P14", "topic": "ci_cd",
     "text": "CI/CD pipelines automate building, testing, and deploying code. Continuous Integration runs tests on every commit to catch errors early. Pipeline stages typically include: linting, unit tests, integration tests, security scanning, artifact building, and deployment. GitHub Actions uses YAML workflow files triggered by events."},
    {"id": "P15", "topic": "ci_cd",
     "text": "Blue-green deployment maintains two identical production environments. Traffic goes to 'blue' while 'green' receives the update. After verification, traffic switches to green. Canary deployment routes 5-10% of traffic to the new version initially. If metrics are healthy, traffic gradually increases to 100%."},
    {"id": "P16", "topic": "monitoring",
     "text": "Prometheus collects time-series metrics using a pull model — it scrapes HTTP endpoints at configured intervals. Metrics types: counters (only increase), gauges (go up and down), histograms (distribution of values). PromQL is the query language. AlertManager handles routing and deduplication of alerts."},
    {"id": "P17", "topic": "monitoring",
     "text": "Distributed tracing tracks requests across microservice boundaries using trace and span IDs. Each span represents a unit of work with start time, duration, and metadata. Jaeger and Zipkin are popular tracing backends. OpenTelemetry provides a vendor-neutral SDK for instrumenting applications."},
    {"id": "P18", "topic": "ml_ops",
     "text": "MLflow tracks machine learning experiments including parameters, metrics, and artifacts. The Model Registry manages model versions and lifecycle stages (staging, production, archived). Models can be served via REST API. MLflow supports Python, R, Java, and integrates with major ML frameworks."},
    {"id": "P19", "topic": "ml_ops",
     "text": "Data drift occurs when production data distribution diverges from training data. Covariate shift changes input feature distributions. Concept drift changes the input-output relationship. The Population Stability Index (PSI) quantifies distribution change. Automated retraining pipelines trigger when drift exceeds a threshold."},
    {"id": "P20", "topic": "architecture",
     "text": "Microservices decompose applications into independently deployable services communicating over networks. Each service owns its data and has its own CI/CD pipeline. Benefits: independent scaling, fault isolation, technology diversity. Challenges: distributed transactions, eventual consistency, operational overhead."},
    {"id": "P21", "topic": "architecture",
     "text": "Event sourcing stores every state change as an immutable event. The current state is derived by replaying all events. This provides a complete audit trail and enables temporal queries. CQRS separates read and write models — commands produce events, queries read from optimized projections."},
    {"id": "P22", "topic": "architecture",
     "text": "The CAP theorem states distributed systems cannot simultaneously guarantee Consistency, Availability, and Partition tolerance. During network partitions, systems must choose: CP systems (HBase, MongoDB with majority reads) reject requests to maintain consistency. AP systems (Cassandra, DynamoDB) accept writes that may conflict."},
    {"id": "P23", "topic": "networking",
     "text": "Load balancers distribute traffic across backend servers. Layer 4 balancers route by IP address and port. Layer 7 balancers inspect HTTP headers and URLs for content-based routing. Algorithms include round-robin, least connections, and weighted distribution. Health checks automatically remove unhealthy servers."},
    {"id": "P24", "topic": "networking",
     "text": "CDNs (Content Delivery Networks) cache content at edge locations worldwide. Static assets (images, CSS, JavaScript) are cached at edge nodes close to users. Cache-Control headers specify maximum age. ETags enable conditional requests. CDN invalidation APIs purge stale content when the origin updates."},
    {"id": "P25", "topic": "testing",
     "text": "The testing pyramid recommends many unit tests (fast, isolated), fewer integration tests (component interactions), and minimal end-to-end tests (full user workflows). Unit tests use mocks to isolate functions. Integration tests verify database queries and API calls work between real components."},
]

print(f"Corpus: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Avg length: {sum(len(p['text'].split()) for p in PASSAGES) // len(PASSAGES)} words")

## 10. Build the Retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("kb")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="kb", metadata={"hnsw:space": "cosine"},
)

texts = [p["text"] for p in PASSAGES]
metas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]
ids = [p["id"] for p in PASSAGES]
vecs = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {collection.count()} passages")


def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "passage_id": results["metadatas"][0][i]["passage_id"],
            "topic": results["metadatas"][0][i]["topic"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        })
    return hits

## 11. RAG Answer Generator

In [ ]:
ANSWER_SYSTEM = """You answer technical questions using the provided passages.
Cite passages as [P01], [P02], etc. Be detailed and informative."""


def generate_answer(question: str, passages: list[dict]) -> str:
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
    prompt = (
        f"QUESTION: {question}\n\n"
        f"PASSAGES:\n{context}\n\n"
        "Provide a detailed answer citing [PXX] sources."
    )
    return ask(prompt, system=ANSWER_SYSTEM, temperature=TEMP_ANSWER)


# Quick test
test_passages = retrieve("What is Docker?", top_k=3)
test_answer = generate_answer("What is Docker?", test_passages)
print(f"Test answer ({len(test_answer)} chars):")
wrap_print(test_answer[:200] + "...")

---

# Part B — Groundedness Checking Pipeline

## 12. Step 1 — Claim Extraction

The first step is to decompose the answer into **atomic claims** — individual factual statements that can each be independently verified.

```
ANSWER: "Docker containers [P01] share the host OS kernel, making
        them faster than VMs. They typically start in under 100ms."

CLAIMS:
  1. "Docker containers share the host OS kernel"           ← verifiable
  2. "Sharing the kernel makes them faster than VMs"        ← verifiable
  3. "They typically start in under 100ms"                  ← verifiable (is 100ms in the source?)
```

In [ ]:
EXTRACT_SYSTEM = """You extract atomic factual claims from an answer.
Each claim should be a single, independently verifiable statement."""

EXTRACT_PROMPT = """Extract all factual claims from this answer.
Break compound sentences into separate claims.
Include any numbers, comparisons, or specific details as separate claims.
Ignore hedging language ("might", "could") — extract the core assertion.

ANSWER: {answer}

Return ONLY a JSON list of claim strings:
["claim 1", "claim 2", "claim 3", ...]"""


def extract_claims(answer: str) -> list[str]:
    """Break an answer into atomic, verifiable claims."""
    resp = ask(
        EXTRACT_PROMPT.format(answer=answer),
        system=EXTRACT_SYSTEM,
        temperature=TEMP_VERIFY,
    )
    result = parse_json(resp)
    if isinstance(result, list):
        return [str(c) for c in result if c]
    # Fallback: split on sentences
    return [s.strip() for s in answer.split(".") if len(s.strip()) > 15]


# Demo
demo_answer = (
    "Docker containers [P01] share the host OS kernel, making them lightweight "
    "and fast. They start in seconds compared to VMs which take minutes. "
    "Docker images use layered file systems where each instruction creates a new layer. "
    "Docker supports up to 500 containers per host by default."
)

demo_claims = extract_claims(demo_answer)
print(f"Answer: {demo_answer[:80]}...")
print(f"\nExtracted {len(demo_claims)} claims:")
for i, c in enumerate(demo_claims, 1):
    print(f"  {i}. {c}")

## 13. Step 2 — Claim Verification

For each claim, check whether any passage **entails** (supports) it. The verifier acts as a strict judge — it only marks a claim as supported if the passage explicitly contains the information.

In [ ]:
VERIFY_SYSTEM = """You are a strict fact-checker. You determine whether a claim
is SUPPORTED, PARTIALLY_SUPPORTED, or NOT_SUPPORTED by the given passages.

Rules:
- SUPPORTED: The passage explicitly states or directly implies the claim
- PARTIALLY_SUPPORTED: The passage covers the topic but not the specific detail
- NOT_SUPPORTED: The passage does not contain information to verify this claim

Be STRICT. If a number, comparison, or specific detail is claimed but not
in the passages, mark it NOT_SUPPORTED even if the general topic is covered."""

VERIFY_PROMPT = """Check if this claim is supported by the passages.

CLAIM: {claim}

PASSAGES:
{context}

Return ONLY JSON:
{{"verdict": "SUPPORTED or PARTIALLY_SUPPORTED or NOT_SUPPORTED",
  "supporting_passage": "PXX or null",
  "evidence": "quote or paraphrase from the passage that supports/contradicts",
  "reasoning": "why this verdict"}}"""


def verify_claim(claim: str, passages: list[dict]) -> dict:
    """Verify a single claim against the source passages."""
    context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
    resp = ask(
        VERIFY_PROMPT.format(claim=claim, context=context),
        system=VERIFY_SYSTEM,
        temperature=TEMP_VERIFY,
    )
    result = parse_json(resp)
    if result and "verdict" in result:
        result["claim"] = claim
        return result
    return {
        "claim": claim,
        "verdict": "NOT_SUPPORTED",
        "supporting_passage": None,
        "evidence": "Could not parse verification response",
        "reasoning": "Fallback: unparseable response",
    }


# Demo: verify the claims from the demo answer
demo_passages = retrieve("Docker containers", top_k=3)
print("Verifying demo claims:")
print("=" * 90)
for claim in demo_claims:
    v = verify_claim(claim, demo_passages)
    icon = {"SUPPORTED": "OK", "PARTIALLY_SUPPORTED": "~~", "NOT_SUPPORTED": "XX"}.get(v["verdict"], "??")
    print(f"  [{icon}] {v['verdict']:>20s} | {claim[:60]}")
    if v["verdict"] != "SUPPORTED":
        print(f"       Reason: {v.get('reasoning', '?')[:70]}")

## 14. Step 3 — Full Groundedness Check

Now combine extraction + verification into a single groundedness pipeline that scores the entire answer.

In [ ]:
def check_groundedness(answer: str, passages: list[dict]) -> dict:
    """Full groundedness check: extract claims, verify each, compute score."""
    claims = extract_claims(answer)
    verifications = []

    for claim in claims:
        v = verify_claim(claim, passages)
        verifications.append(v)

    # Compute scores
    n = len(verifications)
    supported = sum(1 for v in verifications if v["verdict"] == "SUPPORTED")
    partial = sum(1 for v in verifications if v["verdict"] == "PARTIALLY_SUPPORTED")
    unsupported = sum(1 for v in verifications if v["verdict"] == "NOT_SUPPORTED")

    # Groundedness score: full=1.0, partial=0.5, unsupported=0.0
    score = (supported + 0.5 * partial) / n if n > 0 else 0.0

    return {
        "total_claims": n,
        "supported": supported,
        "partially_supported": partial,
        "not_supported": unsupported,
        "groundedness_score": score,
        "verifications": verifications,
        "flagged_claims": [v for v in verifications if v["verdict"] == "NOT_SUPPORTED"],
    }


print("Groundedness pipeline ready")
print("  extract_claims() → verify_claim() × N → check_groundedness()")

## 15. Test Groundedness — Well-Grounded Answer

In [ ]:
q1 = "How does Docker work?"
p1 = retrieve(q1, top_k=3)
a1 = generate_answer(q1, p1)

print(f"Q: {q1}")
print(f"Retrieved: {[p['passage_id'] for p in p1]}")
print(f"\nAnswer:")
wrap_print(a1)

print(f"\n{'='*90}")
print("GROUNDEDNESS CHECK")
print("=" * 90)

g1 = check_groundedness(a1, p1)

for v in g1["verifications"]:
    icon = {"SUPPORTED": "OK", "PARTIALLY_SUPPORTED": "~~", "NOT_SUPPORTED": "XX"}.get(v["verdict"], "??")
    print(f"  [{icon}] {v['claim'][:70]}")

print(f"\nSCORE: {g1['groundedness_score']:.0%} grounded")
print(f"  Supported: {g1['supported']}/{g1['total_claims']}")
print(f"  Partial:   {g1['partially_supported']}/{g1['total_claims']}")
print(f"  Flagged:   {g1['not_supported']}/{g1['total_claims']}")

## 16. Test Groundedness — Deliberately Hallucinated Answer

To test the checker robustly, we'll craft an answer that **looks plausible** but contains specific hallucinated claims mixed with real ones.

In [ ]:
# Deliberately mix real claims with hallucinated ones
hallucinated_answer = (
    "Redis [P05] is an in-memory data structure store that achieves sub-millisecond "
    "latency. It supports strings, lists, sets, sorted sets, and hashes. "
    "Redis was created by Salvatore Sanfilippo in 2009 and is written in C. "
    "It can handle up to 1 million concurrent connections per instance. "
    "Redis Cluster supports up to 1000 nodes for horizontal scaling. "
    "Redis Sentinel handles automatic failover for high availability. "
    "For persistence, Redis offers RDB snapshots and AOF logging. "
    "Redis 7.0 introduced Redis Functions as a replacement for Lua scripting."
)

p_redis = retrieve("Redis in-memory cache", top_k=3)

print("TESTING WITH MIXED REAL + HALLUCINATED ANSWER")
print("=" * 90)
print(f"Passages: {[p['passage_id'] for p in p_redis]}")
print(f"\nAnswer (contains deliberate hallucinations):")
wrap_print(hallucinated_answer)

print(f"\n{'='*90}")
print("GROUNDEDNESS CHECK")
print("=" * 90)

g2 = check_groundedness(hallucinated_answer, p_redis)

for v in g2["verifications"]:
    icon = {"SUPPORTED": "OK", "PARTIALLY_SUPPORTED": "~~", "NOT_SUPPORTED": "XX"}.get(v["verdict"], "??")
    claim_short = v["claim"][:65]
    print(f"  [{icon}] {v['verdict']:>20s} | {claim_short}")
    if v["verdict"] != "SUPPORTED":
        print(f"       → {v.get('reasoning', '?')[:75]}")

print(f"\nSCORE: {g2['groundedness_score']:.0%} grounded")
print(f"  Flagged claims: {g2['not_supported']}")
if g2["flagged_claims"]:
    print(f"\n  FLAGGED (likely hallucinated):")
    for fc in g2["flagged_claims"]:
        print(f"    ✗ {fc['claim'][:75]}")

---

# Part C — Citation Verification

## 17. Citation Checker

Beyond claim-level verification, we can check whether **citations are accurate** — does the cited passage actually support what's attributed to it?

In [ ]:
CITATION_SYSTEM = """You verify whether a citation is accurate. When an answer says
'According to [PXX], claim...', check if passage PXX actually says that."""

CITATION_PROMPT = """The answer attributes this claim to a specific passage.

CLAIM: {claim}
CITED PASSAGE ID: {cited_id}
ACTUAL PASSAGE TEXT: {passage_text}

Does the cited passage actually support this claim?

Return ONLY JSON:
{{"citation_correct": true/false,
  "reasoning": "explanation"}}"""


def extract_citations(answer: str) -> list[dict]:
    """Extract cited passage IDs and the claims they support."""
    # Find all [PXX] references and their surrounding text
    pattern = r'\[?(P\d{2})\]?'
    citations = []
    sentences = re.split(r'(?<=[.!?])\s+', answer)

    for sentence in sentences:
        refs = re.findall(pattern, sentence)
        for ref in refs:
            citations.append({
                "cited_id": ref,
                "claim_text": re.sub(r'\[P\d{2}\]', '', sentence).strip(),
            })
    return citations


def verify_citations(answer: str, passages: list[dict]) -> list[dict]:
    """Verify each citation in the answer."""
    citations = extract_citations(answer)
    passage_map = {p["passage_id"]: p["text"] for p in passages}

    results = []
    for cite in citations:
        pid = cite["cited_id"]
        if pid not in passage_map:
            results.append({
                "cited_id": pid,
                "claim": cite["claim_text"],
                "citation_correct": False,
                "reasoning": f"Passage {pid} was not in the retrieved set",
            })
            continue

        resp = ask(
            CITATION_PROMPT.format(
                claim=cite["claim_text"][:200],
                cited_id=pid,
                passage_text=passage_map[pid][:300],
            ),
            system=CITATION_SYSTEM,
            temperature=TEMP_VERIFY,
        )
        result = parse_json(resp)
        if result:
            result["cited_id"] = pid
            result["claim"] = cite["claim_text"]
            results.append(result)
        else:
            results.append({
                "cited_id": pid,
                "claim": cite["claim_text"],
                "citation_correct": False,
                "reasoning": "Could not parse response",
            })

    return results


print("Citation verifier ready")

In [ ]:
# Test citation verification with a mixed answer
citation_test = (
    "According to [P05], Redis is an in-memory store with sub-millisecond latency. "
    "Redis supports distributed transactions [P05] for ACID compliance. "
    "Cache invalidation [P06] uses TTL or write-through strategies. "
    "Redis Cluster [P03] enables horizontal scaling across nodes."
)

test_passages_cit = retrieve("Redis caching", top_k=5)

print("CITATION VERIFICATION")
print("=" * 90)
print(f"Answer: {citation_test[:80]}...")
print()

cit_results = verify_citations(citation_test, test_passages_cit)
for cr in cit_results:
    status = "CORRECT" if cr.get("citation_correct") else "WRONG"
    icon = "OK" if cr.get("citation_correct") else "XX"
    print(f"  [{icon}] [{cr['cited_id']}] {status:>7s} | {cr['claim'][:55]}")
    if not cr.get("citation_correct"):
        print(f"       → {cr.get('reasoning', '?')[:70]}")

---

# Part D — Full Evaluation

## 18. Test Set — Questions That Invite Hallucination

Some questions naturally lead to more hallucination than others. We'll test with a range.

In [ ]:
EVAL_QUESTIONS = [
    # Easy — answer fully in context
    {"question": "What data types does Redis support?",
     "risk": "low"},
    {"question": "What HTTP methods are used in REST APIs?",
     "risk": "low"},
    {"question": "How does Kubernetes Horizontal Pod Autoscaler work?",
     "risk": "low"},

    # Medium — may extrapolate beyond context
    {"question": "Compare PostgreSQL and MySQL for production use.",
     "risk": "medium"},
    {"question": "What are the performance benchmarks for Redis vs Memcached?",
     "risk": "medium"},
    {"question": "How does Kafka compare to RabbitMQ in terms of throughput?",
     "risk": "medium"},

    # High risk — context lacks specifics, LLM may invent
    {"question": "What companies use Kubernetes in production and what scale do they operate at?",
     "risk": "high"},
    {"question": "What is the maximum number of containers Docker can run on a single host?",
     "risk": "high"},
    {"question": "What are the latest features added in Redis 7.0?",
     "risk": "high"},
]

print(f"Evaluation: {len(EVAL_QUESTIONS)} questions")
print(f"Risk levels: {dict(Counter(q['risk'] for q in EVAL_QUESTIONS))}")

## 19. Run Full Evaluation

In [ ]:
print("Running groundedness evaluation (LLM-intensive)...\n")

eval_results = []

for i, item in enumerate(EVAL_QUESTIONS, 1):
    q = item["question"]
    passages = retrieve(q, top_k=3)
    answer = generate_answer(q, passages)
    ground = check_groundedness(answer, passages)

    eval_results.append({
        "question": q,
        "risk": item["risk"],
        "answer": answer,
        "passages": [p["passage_id"] for p in passages],
        "groundedness": ground,
    })

    print(f"  [{i}/{len(EVAL_QUESTIONS)}] [{item['risk']:>6s} risk] "
          f"Score: {ground['groundedness_score']:.0%} "
          f"({ground['supported']}/{ground['total_claims']} supported, "
          f"{ground['not_supported']} flagged) "
          f"| {q[:45]}")

## 20. Overall Results

In [ ]:
print("GROUNDEDNESS RESULTS")
print("=" * 80)

all_scores = [r["groundedness"]["groundedness_score"] for r in eval_results]
all_flagged = sum(r["groundedness"]["not_supported"] for r in eval_results)
all_claims = sum(r["groundedness"]["total_claims"] for r in eval_results)

print(f"\n  Overall groundedness: {sum(all_scores)/len(all_scores):.0%}")
print(f"  Total claims checked: {all_claims}")
print(f"  Total flagged:        {all_flagged} ({all_flagged/all_claims:.0%})")

print(f"\n  BY RISK LEVEL")
print(f"  {'Risk':<10} {'Avg Score':>12} {'Flagged':>10} {'Claims':>10}")
print(f"  {'-'*45}")
for risk in ["low", "medium", "high"]:
    items = [r for r in eval_results if r["risk"] == risk]
    avg_score = sum(r["groundedness"]["groundedness_score"] for r in items) / len(items)
    flagged = sum(r["groundedness"]["not_supported"] for r in items)
    claims = sum(r["groundedness"]["total_claims"] for r in items)
    print(f"  {risk:<10} {avg_score:>11.0%} {flagged:>10} {claims:>10}")

## 21. Detailed Flagged Claims Report

In [ ]:
print("FLAGGED CLAIMS (NOT SUPPORTED by passages)")
print("=" * 100)

for r in eval_results:
    flagged = r["groundedness"]["flagged_claims"]
    if flagged:
        print(f"\n  Q: {r['question'][:65]}")
        print(f"  Passages: {r['passages']} | Score: {r['groundedness']['groundedness_score']:.0%}")
        for fc in flagged:
            print(f"    ✗ CLAIM: {fc['claim'][:70]}")
            print(f"      REASON: {fc.get('reasoning', '?')[:70]}")

---

# Part E — Hardened Groundedness Pipeline

## 22. Self-Correcting RAG

Instead of just flagging hallucinations, we can **regenerate** the answer with instructions to avoid the unsupported claims.

In [ ]:
REGEN_SYSTEM = """You answer technical questions using ONLY the provided passages.
Cite passages as [P01], [P02], etc.

CRITICAL: Some claims from a previous answer were NOT supported by the passages.
You MUST avoid repeating those claims. Only state facts that appear in the passages.
If the passages don't contain enough information, say so explicitly."""


def rag_with_groundedness(question: str, max_retries: int = 2) -> dict:
    """Generate answer, check groundedness, and regenerate if needed."""
    passages = retrieve(question, top_k=3)
    answer = generate_answer(question, passages)
    ground = check_groundedness(answer, passages)

    attempts = [{"answer": answer, "groundedness": ground}]

    retry = 0
    while ground["not_supported"] > 0 and retry < max_retries:
        retry += 1
        flagged_list = "\n".join(
            f"- {fc['claim']}" for fc in ground["flagged_claims"]
        )

        context = "\n\n".join(f"[{p['passage_id']}] {p['text']}" for p in passages)
        prompt = (
            f"QUESTION: {question}\n\n"
            f"PASSAGES:\n{context}\n\n"
            f"WARNING: These claims from a previous attempt were NOT supported:\n{flagged_list}\n\n"
            "Provide a corrected answer using ONLY facts from the passages. "
            "Do NOT include any of the flagged claims."
        )
        answer = ask(prompt, system=REGEN_SYSTEM, temperature=0.1)
        ground = check_groundedness(answer, passages)
        attempts.append({"answer": answer, "groundedness": ground})

    return {
        "question": question,
        "passages": [p["passage_id"] for p in passages],
        "final_answer": answer,
        "final_groundedness": ground,
        "attempts": len(attempts),
        "history": attempts,
    }


print("Self-correcting RAG pipeline ready")
print("  generate → check → if flagged → regenerate (max 2 retries)")

In [ ]:
# Test self-correcting pipeline on high-risk questions
high_risk_qs = [
    "What are the performance benchmarks for Redis vs Memcached?",
    "What is the maximum number of containers Docker can run on a single host?",
    "What companies use Kubernetes in production and what scale do they operate at?",
]

print("SELF-CORRECTING RAG")
print("=" * 100)

for q in high_risk_qs:
    result = rag_with_groundedness(q, max_retries=2)

    print(f"\n  Q: {q}")
    print(f"  Passages: {result['passages']}")
    print(f"  Attempts: {result['attempts']}")

    for i, att in enumerate(result["history"], 1):
        g = att["groundedness"]
        print(f"    Attempt {i}: {g['groundedness_score']:.0%} grounded "
              f"({g['supported']}/{g['total_claims']} supported, "
              f"{g['not_supported']} flagged)")

    improvement = (
        result["history"][-1]["groundedness"]["groundedness_score"] -
        result["history"][0]["groundedness"]["groundedness_score"]
    )
    if improvement > 0:
        print(f"    Improvement: +{improvement:.0%}")
    elif result["attempts"] == 1:
        print(f"    No retry needed")

## 23. Token-Level Entailment Check

For a faster (embedding-based) groundedness signal, we can measure how much of the answer is semantically close to the passages.

In [ ]:
import numpy as np


def embedding_groundedness(answer: str, passages: list[dict]) -> dict:
    """Fast embedding-based groundedness estimate."""
    # Split answer into sentences
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', answer) if len(s.strip()) > 10]
    if not sentences:
        return {"score": 0.0, "details": []}

    # Embed sentences and passages
    sent_vecs = embeddings.embed_documents(sentences)
    passage_texts = [p["text"] for p in passages]
    pass_vecs = embeddings.embed_documents(passage_texts)

    details = []
    for i, (sent, svec) in enumerate(zip(sentences, sent_vecs)):
        sv = np.array(svec)
        max_sim = 0.0
        best_pid = None
        for j, (pvec, p) in enumerate(zip(pass_vecs, passages)):
            pv = np.array(pvec)
            sim = float(np.dot(sv, pv) / (np.linalg.norm(sv) * np.linalg.norm(pv)))
            if sim > max_sim:
                max_sim = sim
                best_pid = p["passage_id"]

        grounded = max_sim >= 0.5  # threshold
        details.append({
            "sentence": sent,
            "max_similarity": max_sim,
            "closest_passage": best_pid,
            "grounded": grounded,
        })

    grounded_count = sum(1 for d in details if d["grounded"])
    return {
        "score": grounded_count / len(details),
        "total_sentences": len(details),
        "grounded_sentences": grounded_count,
        "details": details,
    }


# Compare embedding-based vs LLM-based groundedness
test_q = "How does Redis work?"
test_p = retrieve(test_q, top_k=3)
test_a = generate_answer(test_q, test_p)

emb_ground = embedding_groundedness(test_a, test_p)
llm_ground = check_groundedness(test_a, test_p)

print("EMBEDDING vs LLM GROUNDEDNESS")
print("=" * 80)
print(f"Q: {test_q}")
print(f"Embedding score: {emb_ground['score']:.0%} ({emb_ground['grounded_sentences']}/{emb_ground['total_sentences']} sentences)")
print(f"LLM score:       {llm_ground['groundedness_score']:.0%} ({llm_ground['supported']}/{llm_ground['total_claims']} claims)")

print(f"\nSentence-level embedding details:")
for d in emb_ground["details"]:
    icon = "OK" if d["grounded"] else "XX"
    print(f"  [{icon}] sim={d['max_similarity']:.3f} [{d['closest_passage']}] | {d['sentence'][:55]}")

---

# Part F — Analysis & Learning Notes

## 24. Comparing Groundedness Approaches

| Approach | Speed | Accuracy | Cost | Best For |
|----------|-------|----------|------|----------|
| **LLM claim-level** | Slow (1 LLM call per claim) | Highest — understands nuance | High | Critical applications (medical, legal) |
| **Embedding similarity** | Fast (no LLM needed) | Moderate — misses subtle fabrications | Low | Screening / pre-filtering |
| **Citation verification** | Medium (1 LLM call per citation) | High for citation accuracy | Medium | Systems that cite specific sources |
| **Self-correcting loop** | Very slow (multiple LLM rounds) | Highest end-to-end quality | Very high | Production-critical answers |

### Key Insight: Groundedness Is Not Binary

Real-world groundedness has gradations:

```
FULLY SUPPORTED:       "Redis uses RDB and AOF for persistence"  → in [P05] verbatim
REASONABLY INFERRED:   "Redis is faster than disk-based stores"  → implied by "sub-millisecond"
EXTRAPOLATED:           "Redis is the fastest cache available"    → passage doesn't compare
FABRICATED:             "Redis supports up to 10TB per instance"  → nowhere in any passage
```

A good verifier distinguishes these levels, not just "grounded" vs "not grounded."

In [ ]:
# Final comparison table across all evaluation questions
print("FINAL REPORT")
print("=" * 100)
print(f"{'Question':<55} {'Risk':>6} {'Score':>7} {'Sup':>5} {'Par':>5} {'Flag':>5}")
print("-" * 100)

for r in eval_results:
    g = r["groundedness"]
    q_short = textwrap.shorten(r["question"], 53, placeholder="...")
    print(f"  {q_short:<53} {r['risk']:>6} {g['groundedness_score']:>6.0%} "
          f"{g['supported']:>5} {g['partially_supported']:>5} {g['not_supported']:>5}")

print(f"\n  AVERAGES")
print(f"  Groundedness:  {sum(all_scores)/len(all_scores):.0%}")
print(f"  Total claims:  {all_claims}")
print(f"  Total flagged: {all_flagged} ({all_flagged/all_claims:.0%} hallucination rate)")

## 25. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Only checking overall "is answer correct" | Misses individual hallucinated claims hidden in a mostly-correct answer | Decompose into atomic claims, verify each independently |
| Using the same LLM for both generation and verification | Same model may have same biases / blind spots | Use a different model or at least different temperature and prompt |
| Treating all unsupported claims as equally bad | "Redis was created in 2009" (unverifiable trivia) vs "this drug treats X" (dangerous) | Weight claims by impact/domain relevance |
| Checking only if citations exist | Citation [P05] existing doesn't mean it supports the claim | Verify that the cited passage actually entails the specific claim |
| Skipping groundedness on low-temperature outputs | Even temp=0 can hallucinate — determinism doesn't mean accuracy | Always verify, especially in high-stakes domains |
| Not testing with deliberately hallucinated answers | You don't know your verifier catches real hallucinations | Create synthetic test cases with known hallucinations |

## 26. Production Improvement Ideas

1. **Confidence thresholds** — below 70% groundedness, show a warning; below 50%, refuse to answer
2. **NLI-based verification** — replace LLM judge with a trained NLI model (RoBERTa-MNLI) for speed
3. **Claim importance weighting** — score claims by specificity; generic claims matter less
4. **Human-in-the-loop escalation** — auto-flag answers below groundedness threshold for human review
5. **Groundedness monitoring dashboard** — track scores across queries over time to detect model drift
6. **Fine-tuned verifier** — train a small model specifically for claim verification in your domain
7. **Streaming verification** — check claims as the LLM generates, stopping early if hallucination detected

## 27. Exercises

### Exercise 1
Implement **claim importance scoring**: use the LLM to rate each claim's specificity (1-5). A claim like "Redis is fast" is low specificity; "Redis uses B-tree indexes" is high. Weight the groundedness score by specificity.

### Exercise 2
Build a **groundedness A/B test**: generate 10 answers at temperature 0.1 vs 0.5 and compare groundedness scores. Quantify how much temperature affects hallucination rate.

### Exercise 3
Add **cross-reference verification**: check whether claims are consistent with each other (not just with passages). If two claims contradict each other, flag both regardless of individual passage support.

### Mini Challenge
Build a **groundedness leaderboard**: run the same questions through different prompt strategies (concise vs detailed, strict citations vs free-form) and rank them by groundedness score. Find the prompt that minimizes hallucinations.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Evaluation: {len(EVAL_QUESTIONS)} questions across 3 risk levels")
print(f"")
print(f"Overall groundedness: {sum(all_scores)/len(all_scores):.0%}")
print(f"Claims checked: {all_claims}")
print(f"Hallucinations detected: {all_flagged} ({all_flagged/all_claims:.0%})")
print(f"")
print(f"Pipeline components built:")
print(f"  1. Claim extractor    — breaks answers into atomic verifiable claims")
print(f"  2. Claim verifier     — LLM judges SUPPORTED / PARTIAL / NOT_SUPPORTED")
print(f"  3. Citation checker   — verifies [PXX] cites actually support claims")
print(f"  4. Embedding checker  — fast similarity-based groundedness screening")
print(f"  5. Self-correcting RAG — regenerates answer removing flagged claims")
print(f"  6. Full pipeline      — extract → verify → score → flag → (regenerate)")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **LLMs hallucinate even with good retrieval** — having the right passage doesn't prevent invented details |
| 2 | **Decompose into claims first** — checking an entire answer as a unit misses individual fabrications |
| 3 | **Verify each claim independently** — a mostly-correct answer can hide one dangerous hallucination |
| 4 | **Citation verification is separate from claim verification** — [P05] being a real passage doesn't mean it supports the claim |
| 5 | **Hallucination risk scales with question difficulty** — questions outside the context invite fabrication |
| 6 | **Lower temperature doesn't eliminate hallucination** — it reduces randomness, not factual accuracy |
| 7 | **Self-correcting loops improve groundedness** — flagging unsupported claims and regenerating works |
| 8 | **Embedding similarity is a useful fast check** — catches obvious fabrications without LLM cost |
| 9 | **Production systems need both fast screening and deep verification** — use embedding check first, LLM for flagged answers |
| 10 | **Groundedness is a spectrum**, not binary — SUPPORTED / PARTIALLY_SUPPORTED / NOT_SUPPORTED captures real nuance |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*